# Continue `upper_ppo_direct_last` on the other controllable scenarios

This notebook loads:

`models/upper_ppo_direct_last/run_20260701_232103/upper_ppo_final.zip`

and continues PPO training scenario-by-scenario on the controllable Jain scenarios **other than** `jain_balance_controllable`.

It uses the same direct-offset / weak-handover-penalty strategy:

- `paper_handover_penalty_weight = 0.0`
- `paper_pingpong_penalty_weight = 0.0`
- `expert_bias_reward_weight = 0.5`
- `max_handovers_per_local_step = 3`
- `dynamic_upper_window = True`

After each scenario finishes, it saves a checkpoint and a per-scenario evaluation trace.


In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

if not Path('train_upper_ppo_3gnb.py').exists():
    os.chdir(Path.cwd().parent)

import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from IPython.display import display

from run_zero_action_baseline import build_args
from train_upper_ppo_3gnb import (
    UpperTrainingCsvCallback,
    evaluate_upper_policy,
    make_env,
    save_learning_curve,
)

BASE_MODEL = Path('models/upper_ppo_direct_last/run_20260701_232103/upper_ppo_final.zip')
assert BASE_MODEL.exists(), BASE_MODEL

RUN_NAME = 'upper_ppo_direct_last_continue_other_scenarios'
RUN_DIR = Path('models') / RUN_NAME / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    'jain_control_urllc',
    'jain_control_mmtc',
    'jain_control_embb_urllc',
    'jain_control_embb_mmtc',
    'jain_control_urllc_mmtc',
    'jain_control_outer_congested',
    'jain_control_mixed',
]

# Treat these as per-scenario budgets, not total budget.
TIMESTEPS_PER_SCENARIO = 10_000
EVAL_EPISODES = 10
SEED = 7
DEVICE = 'cpu'

print('Base model:', BASE_MODEL)
print('Run dir:', RUN_DIR)
print('Scenarios:', SCENARIOS)


In [ ]:
def scenario_args(scenario_name, seed=SEED):
    args = build_args(scenario_name, seed)

    # Same strategy as the direct-offset run: allow movement, do not punish HO/PP in paper_cost.
    args.paper_handover_penalty_weight = 0.0
    args.paper_pingpong_penalty_weight = 0.0
    args.expert_bias_reward_weight = 0.5
    args.expert_bias_csv = Path('results/upper_heuristic_3gnb_baseline/upper_heuristic_3gnb_scenario_summary.csv')
    args.expert_bias_closeness_threshold = 0.95

    # Keep the same PPO/runtime setup used by upper_ppo_direct_last.
    args.learning_rate = 1e-4
    args.ppo_n_steps = 512
    args.ppo_batch_size = 128
    args.ppo_n_epochs = 5
    args.ent_coef = 0.0
    args.log_std_init = -1.0
    args.device = DEVICE
    args.total_timesteps = TIMESTEPS_PER_SCENARIO
    args.eval_episodes = EVAL_EPISODES
    args.log_every = 10
    args.log_flush_every = 100

    # Direct-offset training setup.
    args.max_handovers_per_local_step = 3
    args.dynamic_upper_window = True
    args.max_dynamic_local_steps_per_global = 40
    args.safe_admission = True
    args.dense_window_reward = True
    args.use_progress_reward = False

    # Single scenario for this stage.
    args.training_scenarios = scenario_name
    args.single_training_scenario = scenario_name
    args.curriculum_training = False
    args.block_curriculum_training = False
    args.controllable_type1_training = False
    args.controlled_slice_curriculum = False
    args.scenario_selection = 'cycle'
    return args

def save_stage_config(stage_dir, scenario_name, source_model, saved_model, args, validation):
    payload = {
        'scenario_name': scenario_name,
        'source_model': str(source_model),
        'saved_model': str(saved_model),
        'timesteps': int(TIMESTEPS_PER_SCENARIO),
        'seed': int(SEED),
        'strategy': {
            'paper_handover_penalty_weight': float(args.paper_handover_penalty_weight),
            'paper_pingpong_penalty_weight': float(args.paper_pingpong_penalty_weight),
            'expert_bias_reward_weight': float(args.expert_bias_reward_weight),
            'max_handovers_per_local_step': int(args.max_handovers_per_local_step),
            'dynamic_upper_window': bool(args.dynamic_upper_window),
            'max_dynamic_local_steps_per_global': int(args.max_dynamic_local_steps_per_global),
        },
        'validation': validation,
    }
    (stage_dir / 'stage_config.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return payload


In [ ]:
def train_one_scenario(model, scenario_name, stage_idx, source_model_path):
    args = scenario_args(scenario_name)
    stage_dir = RUN_DIR / f'{stage_idx:02d}_{scenario_name}'
    stage_dir.mkdir(parents=True, exist_ok=True)

    env = make_env(args)
    try:
        model.set_env(env)
        model.learning_rate = float(args.learning_rate)
        model.n_steps = int(args.ppo_n_steps)
        model.batch_size = int(args.ppo_batch_size)
        model.n_epochs = int(args.ppo_n_epochs)

        training_csv = stage_dir / 'training_log.csv'
        best_model_path = stage_dir / 'upper_ppo_best.zip'
        final_model_path = stage_dir / 'upper_ppo_after_scenario.zip'
        callback = UpperTrainingCsvCallback(
            training_csv,
            best_model_path,
            log_every=args.log_every,
            flush_every=args.log_flush_every,
        )

        print(f'\n=== Stage {stage_idx}: {scenario_name} ===')
        print('training_csv:', training_csv)
        model.learn(
            total_timesteps=int(TIMESTEPS_PER_SCENARIO),
            callback=callback,
            reset_num_timesteps=False,
            progress_bar=False,
        )
        model.save(final_model_path)
        save_learning_curve(training_csv, stage_dir / 'learning_curve.png')
    finally:
        env.close()

    eval_env = make_env(args)
    try:
        validation_csv = stage_dir / 'validation_log.csv'
        validation = evaluate_upper_policy(
            model,
            eval_env,
            n_eval_episodes=EVAL_EPISODES,
            validation_csv=validation_csv,
        )
    finally:
        eval_env.close()

    payload = save_stage_config(
        stage_dir,
        scenario_name,
        source_model_path,
        final_model_path,
        args,
        validation,
    )
    print('saved:', final_model_path)
    print('validation:', json.dumps(validation, indent=2))
    return final_model_path, payload


In [ ]:
# Main training chain.
# Re-run this cell to start a fresh chain in RUN_DIR.
model = PPO.load(str(BASE_MODEL), device=DEVICE)
source_model = BASE_MODEL
stage_payloads = []

for stage_idx, scenario_name in enumerate(SCENARIOS, start=1):
    saved_model, payload = train_one_scenario(
        model,
        scenario_name,
        stage_idx,
        source_model,
    )
    stage_payloads.append(payload)
    source_model = saved_model

final_chain_model = RUN_DIR / 'upper_ppo_final_after_all_other_scenarios.zip'
model.save(final_chain_model)
summary_path = RUN_DIR / 'scenario_training_summary.json'
summary_path.write_text(json.dumps(stage_payloads, indent=2), encoding='utf-8')
print('\nFinal chain model:', final_chain_model)
print('Summary:', summary_path)


In [ ]:
# Summary table after training completes.
summary_path = RUN_DIR / 'scenario_training_summary.json'
if summary_path.exists():
    rows = json.loads(summary_path.read_text())
    table = []
    for row in rows:
        val = row.get('validation', {})
        table.append({
            'scenario': row['scenario_name'],
            'saved_model': row['saved_model'],
            'mean_eval_return': val.get('mean_eval_return'),
            'mean_target_load_error_delta': val.get('mean_target_load_error_delta'),
            'mean_handover_count_per_step': val.get('mean_handover_count_per_step'),
            'mean_sla_count': val.get('mean_sla_count'),
            'validation_csv': val.get('validation_csv'),
        })
    display(pd.DataFrame(table))
else:
    print('Run the training cell first.')


## Notes

- The notebook saves one model per scenario under `RUN_DIR / 01_scenario_name`, `02_scenario_name`, etc.
- The next scenario starts from the model saved after the previous scenario.
- `upper_ppo_final_after_all_other_scenarios.zip` is the final chained model after all scenario stages.
- If you want a weak but nonzero handover penalty, change `args.paper_handover_penalty_weight` from `0.0` to something like `0.1` in `scenario_args()`.
